In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch accelerate
print("=======Finshed======")

=======Finshed======


In [3]:
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_58/2451623656.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
model_name = "mistralai/Mistral-Nemo-Instruct-2407" 
tokenizer = AutoTokenizer.from_pretrained(model_name) 
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype=torch.float16, 
    device_map="auto"
) 

def generate_text(prompt, max_length=1500, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device) 
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    ) 
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
# 1. Load the candidate's CV from PDF
pdf_path = "/kaggle/input/datasets/youssef2a2hassan/test-cvs/Jannah Ahmed CV _ Mar 25.pdf"
cv_loader = PyPDFLoader(pdf_path)
cv_docs = cv_loader.load()

# 2. Extract full text directly (ideal for typical 1-3 page CVs)
full_cv_text = "\n\n".join([doc.page_content for doc in cv_docs])

# 3. Split text into manageable chunks for vector search indexing
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100)
cv_chunks = text_splitter.split_documents(cv_docs)

# 4. Create embeddings and store in FAISS vector database
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(cv_chunks, embedding)

# 5. Dynamic Context Selection
# If the CV is small so we can give the entire CV for the model but if not so the use of the RAG become here we will give the
# model the best for chunks that have this words or similar to it (look at the generic_query var)
if len(full_cv_text) < 2000:
    context_text = full_cv_text
else:
    # make an object that will be used to search for the chunks inside the VectorDB
    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    generic_query = "technical skills, work experience, main projects, tools, education, and achievements"
    candidate_context = retriever.invoke(generic_query)
    context_text = "\n".join([doc.page_content for doc in candidate_context])

# A print for debugging the app 
print("CV loaded and vector database successfully indexed!")
print(f"Context length prepared: {len(context_text)} characters.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CV loaded and vector database successfully indexed!
Context length prepared: 7738 characters.


In [ ]:
# Define the Parts of the responce that will be cinverted into JSON format that the model understand
question_schema = ResponseSchema(name="question", description="The interview question asked.") 
answer_schema = ResponseSchema(name="answer", description="The candidate's provided answer.")
score_schema = ResponseSchema(name="score", description="A score out of 10 for the answer.")
missing_schema = ResponseSchema(name="missing_points", description="Key technical points the candidate failed to mention.")
improvement_schema = ResponseSchema(name="improvement_suggestions", description="Actionable feedback to improve the answer.")

response_schemas = [question_schema, answer_schema, score_schema, missing_schema, improvement_schema] 
output_parser = StructuredOutputParser.from_response_schemas(response_schemas) 
format_instructions = output_parser.get_format_instructions() 

# ---------------------------------------------------------
# Template for Generating Dynamic Interview Questions
# ---------------------------------------------------------
question_generation_template = """
You are an expert technical recruiter. Review the following background extracted from a candidate's CV:
{context}

Previously asked questions (DO NOT ASK THESE AGAIN):
{asked_questions}

Based on their specific experiences, projects, and skills, generate ONE new, challenging but fair technical interview question tailored exactly to their background. 
Do not include any conversational filler, greetings, or explanations. Output ONLY the question.
"""

# ---------------------------------------------------------
# Template for Evaluating the Candidate's Answer
# ---------------------------------------------------------
evaluation_template = """
You are an expert technical interviewer evaluating a candidate.
Here is the relevant background from the candidate's CV: {context}

You previously asked the candidate this question: "{interview_question}"
The candidate provided this answer: "{candidate_answer}"

Evaluate the candidate's answer based on technical accuracy, completeness, and communication.
Respond ONLY in JSON format as follows:
{format_instructions}
"""


print("=======Finshed======")

=======Finshed======


In [ ]:
import re

# Helper function to extract the JSON block (placed outside the loop)
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL) 
    if matches:
        return f"```json\n{matches[-1]}\n```" 
    return text

# generated comment with the AI for formating
# ---------------------------------------------------------
# Dynamic Interactive Interview Loop
# ---------------------------------------------------------
print("🚀 Starting the AI Interview Simulator...\n")


# Store the asked question to prevent reQuestioning it again
asked_questions = []

while True:
    # 1. Format the question generation prompt using template from Cell 4
    question_prompt = PromptTemplate(
        template=question_generation_template,
        input_variables=["context", "asked_questions"]
    ).format(
        context=context_text, 
        asked_questions="\n".join(asked_questions) if asked_questions else "None"
    )


    # now here we have given the LLM (nistral the prompt of the question and it will generate a question based on the given context wither the entire CV or the Top 4 chunks)
    print("Reading CV and generating tailored interview question...")
    dynamic_interview_question = generate_text(question_prompt, max_length=5000).strip()
    
    # Store the question so it isn't asked again in the next question generation
    asked_questions.append(dynamic_interview_question)

    # generated comment with the AI for formating
    print("\n==========================================")
    print(f"🤖 AI INTERVIEWER ASKS:\n{dynamic_interview_question}")
    print("==========================================\n")

    # 2. Get the Candidate's Answer Dynamically
    candidate_answer = input("👤 YOUR ANSWER (or type 'exit' to quit): ")
    
    if candidate_answer.lower() == 'exit':
        print("\nEnding interview session. Good luck!")
        break

    print("\nEvaluating your answer... This may take a moment.\n")

    # 3. Evaluate the Answer using template and parser from Cell 4
    eval_prompt = PromptTemplate(
        template=evaluation_template,
        input_variables=["context", "interview_question", "candidate_answer", "format_instructions"]
    ).format(
        context=context_text,
        interview_question=dynamic_interview_question,
        candidate_answer=candidate_answer,
        format_instructions=format_instructions
    ) 

    raw_response = generate_text(eval_prompt, max_length=5000)

    # Parse and Display the Results
    # generated comment with the AI for formating
    json_text = extract_json_block(raw_response) 
    try:
        final_evaluation = output_parser.parse(json_text) 
        print("--- 📊 EVALUATION RESULTS ---")
        print(f"Score: {final_evaluation['score']}/10")
        print(f"Missing Points: {final_evaluation['missing_points']}")
        print(f"Suggestions: {final_evaluation['improvement_suggestions']}\n")
        print("-" * 50 + "\n")
    except Exception as e:
        print("Failed to parse output. Raw response:")
        print(raw_response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🚀 Starting the AI Interview Simulator...

Reading CV and generating tailored interview question...

🤖 AI INTERVIEWER ASKS:
You are an expert technical recruiter. Review the following background extracted from a candidate's CV:
Jannah Ahmed 
DEEP LEARNING AND AI   
Cairo, Egypt | 01558414591 | Jannahahmedsaber18@gmail.com 
 
Summary 
 
Highly motivated Engineering student with a deep passion for AI, deep learning, and electronics. With 
hands-on experience in AI leadership, I have mentored and trained students as a Vice Head at 
SemiColon, preparing content, moderating sessions, and instructing on deep learning concepts. 
Beyond leadership, I have competed in deep learning competitions and completed intensive training, 
including the Digital Egypt Pioneers Initiative and Zewail City's AI & Applications program. These 
experiences have strengthened my expertise in machine learning, neural networks, data science, and 
Python programming. 
My strong foundation in electronics and circuit de

👤 YOUR ANSWER (or type 'exit' to quit):  take a new data set and apply the data to it


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Evaluating your answer... This may take a moment.



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


--- 📊 EVALUATION RESULTS ---
Score: 2/10
Missing Points: The candidate missed explaining the steps involved in fine-tuning, the trade-offs, assumptions, evaluation metrics, and interpretation of results.
Suggestions: Candidates should provide a detailed explanation of the process, including data preprocessing, transfer learning, hyperparameter tuning, and model evaluation. They should also discuss potential challenges and how to address them.

--------------------------------------------------

Reading CV and generating tailored interview question...

🤖 AI INTERVIEWER ASKS:
You are an expert technical recruiter. Review the following background extracted from a candidate's CV:
Jannah Ahmed 
DEEP LEARNING AND AI   
Cairo, Egypt | 01558414591 | Jannahahmedsaber18@gmail.com 
 
Summary 
 
Highly motivated Engineering student with a deep passion for AI, deep learning, and electronics. With 
hands-on experience in AI leadership, I have mentored and trained students as a Vice Head at 
SemiColo

👤 YOUR ANSWER (or type 'exit' to quit):  exit



Ending interview session. Good luck!
